# 31_Video 모델 선정 및 모델 input 분석하기

## 학습목표 
- 1. Video 모델을 선정합니다.
  2. 모델이 필요로 하는 Input 데이터를 분석합니다.

In [1]:
import os
from collections import defaultdict

# 데이터셋 루트 디렉토리 (사용자의 데이터 경로에 맞게 수정)
dataset_path = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_2/' # 실제 데이터 경로 입력

# UCF50 데이터셋 내 폴더(클래스) 목록 가져오기
classes = sorted([cls for cls in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, cls))])
num_classes = len(classes)

print(f"총 클래스 개수: {num_classes}")
print("클래스 목록:")
print(classes)

# 각 클래스별 샘플 개수 확인
class_sample_counts = defaultdict(int)

for cls in classes:
    class_path = os.path.join(dataset_path, cls)
    video_files = [f for f in os.listdir(class_path) if f.endswith(('.avi', '.mp4', '.mov', '.mkv'))]
    class_sample_counts[cls] = len(video_files)

# 결과 출력
print("\n각 클래스별 샘플 개수:")
for cls, count in class_sample_counts.items():
    print(f"{cls}: {count}개")

총 클래스 개수: 15
클래스 목록:
['BaseballPitch', 'Biking', 'Diving', 'Fencing', 'HorseRace', 'JumpRope', 'PlayingGuitar', 'PlayingPiano', 'Punch', 'SalsaSpin', 'Skiing', 'Swing', 'TennisSwing', 'VolleyballSpiking', 'YoYo']

각 클래스별 샘플 개수:
BaseballPitch: 10개
Biking: 10개
Diving: 10개
Fencing: 10개
HorseRace: 10개
JumpRope: 10개
PlayingGuitar: 10개
PlayingPiano: 10개
Punch: 10개
SalsaSpin: 10개
Skiing: 10개
Swing: 10개
TennisSwing: 10개
VolleyballSpiking: 10개
YoYo: 10개


In [8]:
from torch.utils.data import Dataset
from torchvision import transforms
import cv2
import numpy as np

class CustomUCF50Dataset(Dataset):
    def __init__(self, root_dir, transform=None, num_frames=16):
        """
        UCF50 비디오 데이터셋을 PyTorch 데이터셋 클래스로 변환.
        
        :param root_dir: UCF50 데이터셋의 루트 디렉토리
        :param transform: 데이터 전처리 및 Augmentation
        :param num_frames: 샘플당 사용할 프레임 개수
        """
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames

        # 클래스별 디렉토리를 탐색하여 비디오 파일을 리스트에 저장
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        # 모든 비디오 파일의 경로 및 레이블을 리스트에 저장
        self.video_list = []
        for cls in self.classes:
            class_path = os.path.join(root_dir, cls)
            video_files = [f for f in os.listdir(class_path) if f.endswith(('.avi', '.mp4', '.mov', '.mkv'))]

            #print(video_files)
            for video in video_files:
                self.video_list.append((os.path.join(class_path, video), self.class_to_idx[cls]))

        #'C:/Users/jeong/Desktop/OnePM/Projects/Datasets/UCF50/UCF50/BaseballPitch\\v_BaseballPitch_g01_c01.avi'
        #print(self.video_list)

    def __len__(self):
        """ 데이터셋 크기 반환 """
        return len(self.video_list)

    def __getitem__(self, idx):
        """
        비디오 데이터를 읽어 PyTorch Tensor로 변환하여 반환.
        :param idx: 데이터 인덱스
        :return: (프레임 텐서, 레이블)
        """
        video_path, label = self.video_list[idx]

        # 비디오에서 프레임 로드
        frames = self._load_video_frames(video_path, self.num_frames)

        # 변환 적용 (torchvision.transforms 활용)
        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])

        return frames, torch.tensor(label, dtype=torch.long)

    def _load_video_frames(self, video_path, num_frames):
        """
        주어진 비디오에서 num_frames 개의 프레임을 균등한 간격으로 샘플링하여 반환.
        :param video_path: 비디오 파일 경로
        :param num_frames: 가져올 프레임 개수
        :return: [num_frames, H, W, C] 형태의 NumPy 배열 리스트
        """
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames == 0:
            cap.release()
            raise ValueError(f"비디오 {video_path}에서 프레임을 로드할 수 없습니다.")

        # 균등한 간격으로 프레임 샘플링
        frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
        frames = []

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV는 BGR 형식이므로 RGB로 변환
            frame = torch.tensor(frame, dtype=torch.float32) / 255.0  # [H, W, C] 정규화
            frames.append(frame)

        cap.release()

        # 프레임이 부족할 경우 마지막 프레임을 반복하여 채움
        while len(frames) < num_frames:
            frames.append(frames[-1].clone())

        frames = torch.stack(frames, dim=0)  # (num_frames, H, W, C)
        frames = frames.permute(0, 3, 1, 2)  # (num_frames, C, H, W)로 변환

        return frames

In [9]:
# 데이터 변환 설정 (Resizing + ToTensor)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 입력 크기 맞추기
    #transforms.ToTensor(),
])

# 데이터셋 로드
dataset = CustomUCF50Dataset(root_dir=dataset_path, transform=transform, num_frames=16)

# 데이터 샘플 확인
sample_frames, sample_label = dataset[0]
print(f"샘플 데이터 크기: {sample_frames.shape}")  # 예상 출력: [16, 3, 224, 224]
print(f"샘플 라벨 (정수 인코딩): {sample_label}")

샘플 데이터 크기: torch.Size([16, 3, 224, 224])
샘플 라벨 (정수 인코딩): 0


In [11]:
import torchvision.models as models

# MobileNet V2 불러오기 (사전 학습된 가중치 사용 가능)
mobilenet_v2 = models.mobilenet_v2(pretrained=True)

# MobileNet V3 (small & large)
mobilenet_v3_small = models.mobilenet_v3_small(pretrained=True)
mobilenet_v3_large = models.mobilenet_v3_large(pretrained=True)

C:\ProgramData\anaconda3\envs\Book\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\anaconda3\envs\Book\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\jeong/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth
100%|█████████████████████████████████████████████████████████████████████████████| 13.6M/13.6M [00:03<00:00, 3.67MB/s]
C:\ProgramData\anaconda3\envs\Book\Lib\site-packages\torchvision\models\_

In [12]:
import torch.nn as nn

# ✅ MobileNet V2 백본 추출
mobilenet_v2 = models.mobilenet_v2(pretrained=True)
mobilenet_v2_backbone = mobilenet_v2.features  # 분류기 제거

# ✅ MobileNet V3 백본 추출
mobilenet_v3 = models.mobilenet_v3_large(pretrained=True)
mobilenet_v3_backbone = mobilenet_v3.features  # 분류기 제거

In [17]:
import torch
import torch.nn as nn

# ✅ MobileNetV3 백본 추출
mobilenet_v3 = models.mobilenet_v3_large(pretrained=True)
mobilenet_v3_backbone = mobilenet_v3.features  # 백본만 사용

class MobileNetFeatureExtractor(nn.Module):
    """ MobileNetV3에서 Feature Map을 추출하여 LSTM 입력 형태로 변환 """
    def __init__(self, backbone, output_dim=512):
        super().__init__()
        self.backbone = backbone  # ✅ MobileNetV3 CNN 백본
        self.global_pool = nn.AdaptiveAvgPool2d(1)  # ✅ Feature Map을 1x1로 축소
        self.fc = nn.Linear(960, output_dim)  # ✅ MobileNetV3 출력 채널(960)을 LSTM 입력 크기(512)로 변환

    def forward(self, x):
        batch_size, num_frames, channels, height, width = x.shape
        x = x.view(batch_size * num_frames, channels, height, width)  # [B*T, C, H, W]

        # ✅ CNN Backbone을 통해 Feature Map 추출
        features = self.backbone(x)  # [B*T, 960, H', W']
        features = self.global_pool(features)  # [B*T, 960, 1, 1]
        features = features.view(batch_size * num_frames, -1)  # [B*T, 960]

        # ✅ FC Layer를 통해 LSTM 입력 크기로 변환
        features = self.fc(features)  # [B*T, 512]
        features = features.view(batch_size, num_frames, -1)  # [B, T, 512]

        return features  # LSTM 입력 형식으로 변환된 Feature Vector

class ActionClassifier(nn.Module):
    def __init__(self, feature_extractor, num_classes, hidden_dim=256, num_layers=2):
        super().__init__()
        self.feature_extractor = feature_extractor  # ✅ MobileNetV3 기반 Feature Extractor
        self.lstm = nn.LSTM(input_size=512, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)  # ✅ CNN 백본으로 Feature 추출 → [B, T, 512]

        # ✅ LSTM으로 Temporal 정보 학습
        lstm_out, _ = self.lstm(features)  # [B, T, hidden_dim]
        action_logits = self.fc(lstm_out[:, -1, :])  # 마지막 타임스텝의 출력만 사용

        return action_logits  # [B, num_classes]

# ✅ 모델 생성
feature_extractor = MobileNetFeatureExtractor(mobilenet_v3_backbone)
action_classifier = ActionClassifier(feature_extractor, num_classes=10)

# ✅ 더미 입력 생성 (batch_size=2, num_frames=16, channels=3, height=224, width=224)
dummy_input = torch.randn(2, 16, 3, 224, 224)

# ✅ 예측 실행
output = action_classifier(dummy_input)
print("Output Shape:", output.shape)  # 예상 결과: [2, 10] (배치 크기, 클래스 수)

Output Shape: torch.Size([2, 10])
